In [1]:
import os, sys, json
from pathlib import Path

ROOT = Path(r"D:\python\python_code\torch_learnling\LLM_Research\smoldataenvs-win")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["TMP"] = r"D:\Temp"
os.environ["TEMP"] = r"D:\Temp"
print(ROOT)


D:\python\python_code\torch_learnling\LLM_Research\smoldataenvs-win


In [2]:
import pandas as pd

df = pd.read_parquet("data/SmolDataEnvs/data/train-00000-of-00001.parquet")
print("shape:", df.shape)
print("columns:", df.columns.tolist())




shape: (5000, 15)
columns: ['task_id', 'question', 'answer', 'reward_mode', 'atol', 'rtol', 'difficulty_level', 'difficulty_tier', 'kaggle_dataset', 'hf_bucket', 'bucket_prefix', 'files', 'instruction', 'source_row_id', 'package_tier']


In [3]:
df[["task_id", "question", "answer", "reward_mode",
    "difficulty_tier", "bucket_prefix", "files"]].head(3)

,task_id,question,answer,reward_mode,difficulty_tier,bucket_prefix,files
0,0000_324_324276_qa_3,What is the most common job role interest amon...,Web Development,flexible,easy,freecodecamp__2016-new-coder-survey-,[2016-FCC-New-Coders-Survey-Data.csv]
1,0000_369_369503_qa_1,What percentage of all matches have a goal dif...,25.4%,flexible,medium,hugomathien__soccer,[database.sqlite]
2,0000_455_455459_qa_4,What is the error rate (as a percentage) for n...,2,flexible,hard,abcsds__pokemon,[Pokemon.csv]


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   task_id           5000 non-null   object 
 1   question          5000 non-null   object 
 2   answer            5000 non-null   object 
 3   reward_mode       5000 non-null   object 
 4   atol              5000 non-null   float64
 5   rtol              5000 non-null   float64
 6   difficulty_level  5000 non-null   int64  
 7   difficulty_tier   5000 non-null   object 
 8   kaggle_dataset    5000 non-null   object 
 9   hf_bucket         5000 non-null   object 
 10  bucket_prefix     5000 non-null   object 
 11  files             5000 non-null   object 
 12  instruction       5000 non-null   object 
 13  source_row_id     5000 non-null   object 
 14  package_tier      5000 non-null   int64  
dtypes: float64(2), int64(2), object(11)
memory usage: 586.1+ KB


In [6]:
def staged(r):
    files = list(r["files"])
    return len(files) > 0 and all(
        (ROOT / "data/task_files" / r["bucket_prefix"] / f).exists() for f in files)

row = next(r for r in df.to_dict("records") if staged(r))
row["files"] = list(row["files"])

for k in ("task_id", "question", "answer", "reward_mode", "atol", "rtol",
          "difficulty_tier", "kaggle_dataset", "hf_bucket", "bucket_prefix", "files"):
    print(f"{k}: {row[k]!r}")
print("\ninstruction:\n", row["instruction"])


task_id: '0000_324_324276_qa_3'
question: 'What is the most common job role interest among new coders?'
answer: 'Web Development'
reward_mode: 'flexible'
atol: 0.0
rtol: 0.0
difficulty_tier: 'easy'
kaggle_dataset: 'freecodecamp/2016-new-coder-survey-'
hf_bucket: 'AdithyaSK/jupyter-agent-kaggle-all'
bucket_prefix: 'freecodecamp__2016-new-coder-survey-'
files: ['2016-FCC-New-Coders-Survey-Data.csv']

instruction:
 You are a data-analysis agent working in a sandbox. Use your code-execution tool to inspect the files and compute the answer.

Files (in /home/user/input, no subfolders):
- 2016-FCC-New-Coders-Survey-Data.csv

Installed: pandas, numpy, matplotlib, seaborn, scipy, scikit-learn, statsmodels, tabulate, sqlite3, plotly (pip install more if needed).

Question:
What is the most common job role interest among new coders?

Work it out step by step — inspect the data first (head, shape, dtypes), then compute.

Answer with a single clean value: a bare number (no commas or units, e.g. 952

In [7]:
prefix = ROOT / "data/task_files" / row["bucket_prefix"]
for f in row["files"]:
    p = prefix / f
    print(f"{f}: exists={p.exists()} size={p.stat().st_size if p.exists() else '-'}")

# 预览第一个文件（csv / tsv 直接读前 5 行；其它格式看前几行文本）
p = prefix / row["files"][0]
if p.suffix.lower() in (".csv", ".tsv"):
    display(pd.read_csv(p, nrows=5, sep="\t" if p.suffix.lower() == ".tsv" else ","))
else:
    print(p.read_text(errors="replace")[:500])


2016-FCC-New-Coders-Survey-Data.csv: exists=True size=10079792


,Age,AttendedBootcamp,BootcampFinish,BootcampFullJobAfter,BootcampLoanYesNo,BootcampMonthsAgo,BootcampName,BootcampPostSalary,BootcampRecommend,ChildrenNumber,...,ResourceSoloLearn,ResourceStackOverflow,ResourceTreehouse,ResourceUdacity,ResourceUdemy,ResourceW3Schools,ResourceYouTube,SchoolDegree,SchoolMajor,StudentDebtOwe
0,28,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"some college credit, no degree",NaN,20000.0
1,22,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1.0,NaN,NaN,"some college credit, no degree",NaN,NaN
2,19,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,high school diploma or equivalent (GED),NaN,NaN
3,26,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,bachelor's degree,Cinematography And Film,7000.0
4,20,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"some college credit, no degree",NaN,NaN


In [8]:
listing = json.loads((ROOT / "data/bucket_listing.json").read_text(encoding="utf-8"))
key = f"{row['hf_bucket']}::{row['bucket_prefix']}"
entries = listing.get(key, [])
for e in entries:
    if Path(e["path"]).name in row["files"]:
        print(e["path"], e["size"])


freecodecamp__2016-new-coder-survey-/2016-FCC-New-Coders-Survey-Data.csv 10079792


In [9]:
sft = pd.read_parquet("data/SmolDataEnvs-sft/data/train-00000-of-00001.parquet")
print("SFT shape:", sft.shape, "columns:", sft.columns.tolist())

r = sft.iloc[0]
msgs = [json.loads(m) for m in r.messages]
print("tools:", len(r.tools), "| tool[0].function.name:", r.tools[0]["function"]["name"])
print("roles:", [m["role"] for m in msgs])

for i, m in enumerate(msgs[:8]):
    c = (m.get("content") or "")
    print(f"\n[{i}] {m['role']}: {c[:300]}")
    if m.get("tool_calls"):
        for tc in m["tool_calls"]:
            print("    tool_call:", tc["function"])


SFT shape: (4677, 7) columns: ['task_id', 'difficulty', 'difficulty_tier', 'n_turns', 'source_agent', 'messages', 'tools']
tools: 1 | tool[0].function.name: bash
roles: ['system', 'user', 'assistant', 'tool', 'assistant', 'tool', 'assistant', 'tool', 'assistant', 'tool', 'assistant', 'tool', 'assistant', 'tool', 'assistant', 'tool', 'assistant']

[0] system: You are an autonomous data-analysis agent operating in a sandboxed Linux container. Your only tool is `bash`. The dataset files are in /home/user/input/. Python 3 + pandas + numpy + scikit-learn + scipy are pre-installed.

Work the problem step-by-step: first inspect the data (ls, head, shape, dtype

[1] user: You are a data-analysis agent working in a sandbox. Use your code-execution tool to inspect the files and compute the answer.

Files (in /home/user/input, no subfolders):
- bank.csv

Installed: pandas, numpy, matplotlib, seaborn, scipy, scikit-learn, statsmodels, tabulate, sqlite3, plotly (pip insta

[2] assistant: I'll analy

In [10]:
import smolenv_local as env

msgs = env.agent_messages(row)
print("=== SYSTEM ===\n", msgs[0]["content"])
print("\n=== USER ===\n", msgs[1]["content"][:800])
print("\n=== TOOL SCHEMA ===")
print(json.dumps(env.BASH_TOOL, ensure_ascii=False, indent=1))


=== SYSTEM ===
 You are an autonomous data-analysis agent operating in a sandboxed Linux container. Your only tool is `bash`. The dataset files are in /home/user/input/. Python 3 + pandas + numpy + scikit-learn + scipy are pre-installed.

Work the problem step-by-step: first inspect the data (ls, head, shape, dtypes), then plan, then compute, then submit.

To submit your final answer you MUST call the `bash` tool to write it to /workdir/answer.txt, e.g. `echo -n "<value>" > /workdir/answer.txt`. Keep the answer short and concise. Do NOT end your turn without submitting.

=== USER ===
 You are a data-analysis agent working in a sandbox. Use your code-execution tool to inspect the files and compute the answer.

Files (in /home/user/input, no subfolders):
- 2016-FCC-New-Coders-Survey-Data.csv

Installed: pandas, numpy, matplotlib, seaborn, scipy, scikit-learn, statsmodels, tabulate, sqlite3, plotly (pip install more if needed).

Question:
What is the most common job role interest among ne

In [12]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("models/Qwen3-0.6B")

text = tok.apply_chat_template(
    msgs, tools=[env.BASH_TOOL], tokenize=False,
    add_generation_prompt=True, enable_thinking=False)
print(text)


<|im_start|>system
You are an autonomous data-analysis agent operating in a sandboxed Linux container. Your only tool is `bash`. The dataset files are in /home/user/input/. Python 3 + pandas + numpy + scikit-learn + scipy are pre-installed.

Work the problem step-by-step: first inspect the data (ls, head, shape, dtypes), then plan, then compute, then submit.

To submit your final answer you MUST call the `bash` tool to write it to /workdir/answer.txt, e.g. `echo -n "<value>" > /workdir/answer.txt`. Keep the answer short and concise. Do NOT end your turn without submitting.

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "bash", "description": "Run a shell command in the sandbox and return its combined stdout+stderr. The shell is non-stateful between calls (each call is a fresh `bash -c`). Use it to explore files (ls, head, cat), run P

In [13]:
sb = env.LocalSandbox(
    row,
    data_root=ROOT / "data/task_files",
    work_parent=ROOT / "runs" / "notebook-demo",
    cfg=env.SandboxConfig(timeout_s=60, keep=True),
)
print("backend:", sb.backend)
print("missing files:", sb.missing_files)
print("sandbox dir:", sb.root)


backend: win-docker
missing files: []
sandbox dir: D:\python\python_code\torch_learnling\LLM_Research\smoldataenvs-win\runs\notebook-demo\0000_324_324276_qa_3-c72de1d5


In [14]:
res = sb.run_bash("pwd; echo '---'; ls -la /home/user/input")
print("rc:", res.returncode, "| timed_out:", res.timed_out, "| seconds:", round(res.seconds, 2))
print("STDOUT+STDERR:\n", (res.stdout + res.stderr))


rc: 0 | timed_out: False | seconds: 0.47
STDOUT+STDERR:
 /workdir
---
total 9848
drwxrwxrwx 1 root root     4096 Nov 12 06:58 .
drwxr-xr-x 3 root root     4096 Nov 12 07:01 ..
-rwxrwxrwx 2 root root 10079792 Nov 11 09:11 2016-FCC-New-Coders-Survey-Data.csv



In [15]:
fname = row["files"][0]
code = f"""
import pandas as pd
p = '/home/user/input/{fname}'
df = pd.read_csv(p)
print('shape:', df.shape)
print('columns:', list(df.columns)[:10])
print(df.head(3))
"""
res = sb.run_python(code)
print("rc:", res.returncode)
print(res.stdout)
print("STDERR:", res.stderr[:500])


rc: 0
shape: (15620, 113)
columns: ['Age', 'AttendedBootcamp', 'BootcampFinish', 'BootcampFullJobAfter', 'BootcampLoanYesNo', 'BootcampMonthsAgo', 'BootcampName', 'BootcampPostSalary', 'BootcampRecommend', 'ChildrenNumber']
    Age  AttendedBootcamp  ...  SchoolMajor  StudentDebtOwe
0  28.0               0.0  ...          NaN         20000.0
1  22.0               0.0  ...          NaN             NaN
2  19.0               0.0  ...          NaN             NaN

[3 rows x 113 columns]

STDERR: /workdir/.prog02.py:4: DtypeWarning: Columns (21: CodeEventOther, 57: JobRoleInterestOther) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(p)



In [16]:
out, res = env.run_tool_call(sb, "bash", {"command": "ls /home/user/input"})
print("tool message content:")
print(out[:500])


tool message content:
2016-FCC-New-Coders-Survey-Data.csv



In [17]:
import shlex
sb.run_bash(f"echo -n {shlex.quote(str(row['answer']))} > /workdir/answer.txt")
answer = sb.answer_txt()
print("answer.txt:", repr(answer))
print("score:", env.score(row, answer))


answer.txt: 'Web Development'
score: {'reward_v1': 1.0, 'method_v1': 'exact', 'reward_v2': 1.0, 'reason_v2': 'exact'}


In [18]:
from eval_harness import LocalHFClient

client = LocalHFClient("models/Qwen3-0.6B")
messages = env.agent_messages(row)

for turn in range(3):
    resp = client.chat(messages, tools=[env.BASH_TOOL],
                       max_tokens=256, temperature=0.8)
    print(f"\n--- model turn {turn+1} ---")
    print("content:", resp["content"][:300])
    print("tool_calls:", resp["tool_calls"])
    if not resp["tool_calls"]:
        break

    messages.append({"role": "assistant", "content": resp["content"],
                     "tool_calls": resp["tool_calls"]})
    for tc in resp["tool_calls"]:
        out, _ = env.run_tool_call(sb, tc["function"]["name"], tc["function"]["arguments"])
        print("tool output:", out[:300])
        messages.append({"role": "tool", "tool_call_id": tc["id"],
                         "name": tc["function"]["name"], "content": out})

print("\nfinal answer.txt:", repr(sb.answer_txt()))


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]


--- model turn 1 ---
content: 
tool_calls: [{'id': 'call_0', 'type': 'function', 'function': {'name': 'bash', 'arguments': '{"command": "ls /home/user/input && head -n 50 /home/user/input/2016-FCC-New-Coders-Survey-Data.csv && cat /home/user/input/2016-FCC-New-Coders-Survey-Data.csv"}'}}]
tool output: 2016-FCC-New-Coders-Survey-Data.csv
"Age","AttendedBootcamp","BootcampFinish","BootcampFullJobAfter","BootcampLoanYesNo","BootcampMonthsAgo","BootcampName","BootcampPostSalary","BootcampRecommend","ChildrenNumber","CityPopulation","CodeEventBootcamp","CodeEventCoffee","CodeEventConferences","CodeEve

--- model turn 2 ---
content: Here is a summary of the data provided, highlighting the key information for each entry:

### Fields Overview:
- **ID**: Unique identifier for each entry
- **Name**: Full name
- **Nationality**: Country of residence
- **Field**: Profession
- **Education**: Type of education
- **Employment Status**: 
tool_calls: []

final answer.txt: 'Web Development'
